In [4]:
import nest_asyncio
nest_asyncio.apply()

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
from flask import Flask, request, render_template_string
import base64

# ===== LOAD MODEL =====
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("model.pt", map_location='cpu'))
model.eval()

# ===== TRANSFORM =====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ===== HTML UI ĐẸP =====
HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <title>Dog Detector</title>
    <style>
        body {
            background: #343541;
            color: white;
            font-family: Arial;
            text-align: center;
        }
        .box {
            margin: 50px auto;
            background: #444654;
            padding: 30px;
            border-radius: 15px;
            width: 400px;
        }
        .upload {
            border: 2px dashed #19c37d;
            padding: 20px;
            border-radius: 10px;
            cursor: pointer;
            display: inline-block;
        }
        .upload:hover {
            background: #3a3b47;
        }
        input { display: none; }
        button {
            margin-top: 15px;
            padding: 10px 20px;
            background: #19c37d;
            border: none;
            border-radius: 8px;
            color: white;
            cursor: pointer;
        }
        img {
            margin-top: 20px;
            max-width: 100%;
            border-radius: 10px;
        }
    </style>
</head>
<body>

<h1>🐶 Dog Detector</h1>

<div class="box">
    <form method="POST" enctype="multipart/form-data">
        <label class="upload">
            📁 Chọn ảnh
            <input type="file" name="file" required>
        </label>
        <br>
        <button type="submit">Dự đoán</button>
    </form>

    {% if prediction %}
        <h2>{{ prediction }}</h2>
        <img src="{{ image }}">
    {% endif %}
</div>

</body>
</html>
"""

# ===== FLASK =====
app = Flask(__name__)

@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'POST':
        file = request.files['file']

        if file:
            # ===== convert ảnh sang base64 để hiển thị =====
            image_bytes = file.read()
            encoded_img = base64.b64encode(image_bytes).decode('utf-8')
            img_data = f"data:image/jpeg;base64,{encoded_img}"

            # ===== xử lý cho model =====
            image = Image.open(file.stream).convert("RGB")
            image_tensor = transform(image).unsqueeze(0)

            with torch.no_grad():
                outputs = model(image_tensor)
                probs = F.softmax(outputs, dim=1)

                dog_prob = probs[0][0].item()
                _, predicted = torch.max(outputs, 1)

            if predicted.item() == 0 and dog_prob > 0.5:
                result = f"🐶 Đây là CHÓ ({dog_prob:.2f})"
            else:
                result = f"❌ Không phải chó ({dog_prob:.2f})"

            return render_template_string(
                HTML_PAGE,
                prediction=result,
                image=img_data
            )

    return render_template_string(HTML_PAGE)

In [6]:
app.run(port=5000, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [07/Apr/2026 11:00:44] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 11:00:50] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 11:00:58] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 11:01:04] "POST / HTTP/1.1" 200 -
